# 🎓 UniGuide AI: Enterprise Agentic Admissions Advisor
### Production-Ready LangGraph Architecture with Hybrid RAG & Conversational Memory

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)
[![Powered by Groq](https://img.shields.io/badge/LLM-Groq%20Cloud-orange)](https://groq.com/)
[![LangGraph](https://img.shields.io/badge/LangGraph-0.2%2B-blue)](https://github.com/langchain-ai/langgraph)
[![FAISS Vector Store](https://img.shields.io/badge/VectorStore-FAISS-green)](https://github.com/facebookresearch/faiss)
[![Stateful Checkpointer](https://img.shields.io/badge/Memory-MemorySaver-purple)](https://langchain-ai.github.io/langgraph/)

---

## 📌 Executive Summary
**UniGuide AI** is an intelligent university admissions consultant built for **University of Education (UE)**. It utilizes a **ReAct (Reasoning + Acting) Agentic Workflow** orchestrated via **LangGraph**, powered by **Groq Cloud**:
1. **Dynamic Model Discovery**: Automatically queries your Groq account to find active models (`openai/gpt-oss-20b`, `openai/gpt-oss-120b`, etc.) preventing any 404 model deprecation errors.
2. **Auto-Retry Rate Limit Protection**: Built-in exponential backoff handling Groq free-tier TPM (Tokens Per Minute) smoothly without rate-limit interrupts.
3. **Dynamic PDF RAG Engine**: Semantic document search over the official `HIST_Admissions_Knowledge_Base.pdf` using HuggingFace embeddings (`all-MiniLM-L6-v2`) and FAISS.
4. **Safe Mathematical Tooling**: AST-based computational tool for precise calculation of semester fees, merit scholarships, need-based aid, and sibling discounts without arithmetic hallucination.
5. **Conversational Memory**: Checkpointed state persistence (`MemorySaver`) maintaining multi-turn conversational context across unique `thread_id` sessions.
6. **Guardrail-Enforced System Guidance**: Zero-hallucination policy ensuring strict adherence to official institutional guidelines.

---

## 🏗️ Architecture Blueprint

```
+---------------------------------------------------------------------------------+
|                                STUDENT USER                                     |
+---------------------------------------------------------------------------------+
                                         |
                                         v
                         +-------------------------------+
                         |   LangGraph Agent State       |
                         |   (Messages + Session State)  |
                         +-------------------------------+
                                         |
                                         +<=============> [ MemorySaver Checkpointer ]
                                         |                (Thread-scoped Session Memory)
                                         v
                         +-------------------------------+
                         |      Groq Cloud LLM           |
                         |      (LLM Reasoning Cycle)    |
                         +-------------------------------+
                                         |
                     +-------------------+-------------------+
                     | (Needs Tool)                          | (Direct Answer)
                     v                                       v
         +-----------------------+                    +--------------+
         |      ToolNode         |                    |  Final User  |
         +-----------------------+                    |   Response   |
           /         |         \                     +--------------+
          /          |          \
         v           v           v
  +-------------+ +----------+ +-------------------------+
  |  FAISS RAG  | |   Safe   | | Structured Fast Lookup  |
  |  (PDF Docs) | |   Math   | |  (In-Memory Catalog)    |
  +-------------+ +----------+ +-------------------------+
```

## 📦 Module 1: Unified Environment Setup & Installation
Install all required production libraries in a single step. This configuration runs seamlessly on both **Google Colab** and **Local Environments (Windows, macOS, Linux)**.

In [ ]:
# Comprehensive dependency installation
!pip install -q --upgrade langgraph langchain langchain-core langchain-community langchain-groq groq langchain-google-genai langchain-openai langchain-ollama langchain-huggingface sentence-transformers faiss-cpu pypdf

print("All dependencies installed successfully!")

## ⚡ Module 2: Bulletproof Groq LLM Configuration
This cell connects to Groq and **automatically detects the active models available on your specific account** (e.g. `openai/gpt-oss-20b`, `openai/gpt-oss-120b`) with built-in retry mechanisms.

In [ ]:
import os
import sys
import getpass
from groq import Groq
from langchain_groq import ChatGroq

# 1. Acquire Groq API Key
groq_key = os.environ.get("GROQ_API_KEY", "").strip()
if not groq_key:
    try:
        from google.colab import userdata
        groq_key = userdata.get("GROQ_API_KEY", "").strip()
    except Exception:
        pass

if not groq_key:
    groq_key = input("Enter your Groq API Key (from https://console.groq.com/keys): ").strip()

os.environ["GROQ_API_KEY"] = groq_key

# 2. Automatically query your Groq account for currently active models
client = Groq(api_key=groq_key)
try:
    available_models = [
        m.id for m in client.models.list().data 
        if "whisper" not in m.id.lower() and "guard" not in m.id.lower()
    ]
    print(f"[MODELS] Active models on your Groq account ({len(available_models)} found):", available_models)
    
    # Priority list of current active models on Groq:
    priority = [
        "openai/gpt-oss-20b",
        "openai/gpt-oss-120b",
        "llama-3.3-70b-versatile",
        "llama-3.1-8b-instant",
        "llama3-70b-8192",
        "llama3-8b-8192",
        "gemma2-9b-it"
    ]
    
    selected_model = None
    for p in priority:
        if p in available_models:
            selected_model = p
            break
            
    if not selected_model and available_models:
        selected_model = available_models[0]
        
except Exception as e:
    print(f"[NOTE]: {e}")
    selected_model = "openai/gpt-oss-20b"

print(f"[SELECTED] Active Model: '{selected_model}'")

# 3. Initialize LangChain Groq instance with automatic retries on rate limits
llm = ChatGroq(
    model_name=selected_model,
    temperature=0.0,
    max_retries=5,
    timeout=60
)

# 4. Immediate live test call to ensure 100% working connection
test_out = llm.invoke("Respond with the word 'READY'")
print(f"[SUCCESS] Groq Connection Verified! Response: {test_out.content.strip()}")

## 📚 Module 3: Knowledge Base RAG Pipeline (HIST Admissions PDF)
This module loads the official **University of Education (UE)** admissions handbook:
1. **Intelligent PDF Discovery**: Detects the local `HIST_Admissions_Knowledge_Base.pdf` file or falls back to an upload widget on Google Colab.
2. **Text Chunking**: Uses `RecursiveCharacterTextSplitter` with 500-token chunk size and 100-token overlap to maintain semantic continuity across policy clauses.
3. **Dense Vector Embeddings**: Uses `sentence-transformers/all-MiniLM-L6-v2` for embeddings.
4. **FAISS Vector Index**: Builds an in-memory vector database with similarity search.

In [ ]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# 1. Locate PDF file
PDF_PATH = "HIST_Admissions_Knowledge_Base.pdf"

if not os.path.exists(PDF_PATH):
    # Try parent or Colab file upload
    if os.path.exists(os.path.join("..", PDF_PATH)):
        PDF_PATH = os.path.join("..", PDF_PATH)
    else:
        try:
            from google.colab import files
            print("PDF not found locally. Please upload HIST_Admissions_Knowledge_Base.pdf:")
            uploaded = files.upload()
            PDF_PATH = list(uploaded.keys())[0]
        except Exception:
            raise FileNotFoundError("Could not find HIST_Admissions_Knowledge_Base.pdf. Place it in the working directory.")

print(f"Loading document from: {PDF_PATH}")
loader = PyPDFLoader(PDF_PATH)
raw_documents = loader.load()
print(f"Loaded {len(raw_documents)} pages successfully.")

# 2. Chunking with semantic overlap
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " "]
)
chunks = text_splitter.split_documents(raw_documents)
print(f"Generated {len(chunks)} text chunks.")

# 3. Embedding Model
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# 4. Create FAISS Vector Store
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print("FAISS Vector Index initialized and ready for semantic retrieval!")

## 🛠️ Module 4: Enterprise Tool Suite
UniGuide AI is equipped with 3 specialized tools:
1. `university_rag_search`: Queries the indexed PDF knowledge base and returns context with source page citations.
2. `safe_calculator`: An AST-based mathematical evaluator that handles tuition fees, percentage discounts, and arithmetic safely without security vulnerabilities or syntax errors.
3. `university_program_lookup`: Fast in-memory structured catalog for quick verification of degree offerings.

In [ ]:
import ast
import operator
import re
from langchain_core.tools import tool

# -------------------------------------------------------------
# Tool 1: High-Precision RAG Retrieval Tool
# -------------------------------------------------------------
@tool
def university_rag_search(query: str) -> str:
    """Search the official HIST university admissions PDF for eligibility criteria,
    fee structures, scholarship rules, admission deadlines, hostel, and contact details."""
    results = retriever.invoke(query)
    if not results:
        return "No relevant information found in the HIST Admissions Knowledge Base."
    
    formatted_sources = []
    for i, doc in enumerate(results):
        page_num = doc.metadata.get("page", "Unknown")
        formatted_sources.append(f"[Source {i+1} - Page {page_num}]:\n{doc.page_content.strip()}")
    
    return "\n\n".join(formatted_sources)


# -------------------------------------------------------------
# Tool 2: AST-Safe Calculator Tool (Handles currency & formulas)
# -------------------------------------------------------------
_ALLOWED_OPERATORS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.Pow: operator.pow,
    ast.USub: operator.neg,
    ast.UAdd: operator.pos,
}

def _eval_ast_node(node):
    if isinstance(node, ast.Constant):
        return node.value
    elif isinstance(node, ast.BinOp):
        left = _eval_ast_node(node.left)
        right = _eval_ast_node(node.right)
        op = _ALLOWED_OPERATORS.get(type(node.op))
        if op is None:
            raise ValueError(f"Unsupported operator: {type(node.op)}")
        return op(left, right)
    elif isinstance(node, ast.UnaryOp):
        operand = _eval_ast_node(node.operand)
        op = _ALLOWED_OPERATORS.get(type(node.op))
        if op is None:
            raise ValueError(f"Unsupported unary operator: {type(node.op)}")
        return op(operand)
    else:
        raise ValueError(f"Unsupported AST node: {type(node)}")

@tool
def safe_calculator(expression: str) -> str:
    """Safely evaluate mathematical expressions for fees, scholarship deductions,
    percentages, and totals. Example inputs: '85000 * 0.50', '85000 - 8500', '(85000 + 20000) * 0.90'."""
    try:
        # Pre-process: remove currency tokens, commas, and trailing symbols
        clean_expr = expression.replace("PKR", "").replace("pkr", "").replace(",", "").replace("$", "").strip()
        # Convert percentages like '15%' to '(15/100)'
        clean_expr = re.sub(r'(\d+(?:\.\d+)?)\s*%', r'(\1 / 100.0)', clean_expr)
        
        tree = ast.parse(clean_expr, mode='eval')
        result = _eval_ast_node(tree.body)
        
        if isinstance(result, float) and result.is_integer():
            return str(int(result))
        return f"{result:,.2f}"
    except Exception as e:
        return f"Calculation error: Could not evaluate '{expression}'. Please provide a standard arithmetic expression."


# -------------------------------------------------------------
# Tool 3: Fast In-Memory Program Catalog
# -------------------------------------------------------------
HIST_CATALOG = {
    "bs cs": {"name": "BS Computer Science", "duration": "4 Years (8 Semesters)", "eligibility": "Intermediate (Pre-Eng/ICS) or A-Levels (with Math) >= 50%", "field": "Computing"},
    "bs ai": {"name": "BS Artificial Intelligence", "duration": "4 Years (8 Semesters)", "eligibility": "Intermediate (Pre-Eng/ICS) or A-Levels (with Math) >= 50%", "field": "Artificial Intelligence"},
    "bs se": {"name": "BS Software Engineering", "duration": "4 Years (8 Semesters)", "eligibility": "Intermediate (Pre-Eng/ICS) or A-Levels (with Math) >= 50%", "field": "Software Engineering"},
    "bba":   {"name": "BS Business Administration", "duration": "4 Years (8 Semesters)", "eligibility": "Intermediate (any discipline) or A-Levels >= 45%", "field": "Business"},
    "ms cs": {"name": "MS Computer Science", "duration": "2 Years (4 Semesters)", "eligibility": "4-year BS (CS/SE/IT) with CGPA >= 2.5/4.0 or 60% marks", "field": "Graduate Computing"}
}

@tool
def university_program_lookup(program_key: str) -> str:
    """Quick lookup for basic degree duration and eligibility for BSCS, BSAI, BSSE, BBA, MSCS."""
    key = program_key.lower().strip()
    for prog, info in HIST_CATALOG.items():
        if prog in key or key in prog or prog.replace(" ", "") in key.replace(" ", ""):
            return (f"Program: {info['name']}\n"
                    f"Duration: {info['duration']}\n"
                    f"Field: {info['field']}\n"
                    f"Eligibility Summary: {info['eligibility']}")
    return "Program not found in quick catalog. Use university_rag_search to search the complete PDF."

# Tool registry
tools = [university_rag_search, safe_calculator, university_program_lookup]
print(f"Registered {len(tools)} production tools: {[t.name for t in tools]}")

## 🧠 Module 5: LangGraph ReAct Workflow with Conversational Memory
In this module, we construct the stateful **LangGraph ReAct Agent**.

### Key Architectural Pillars:
1. **`AgentState`**: Manages message history using `Annotated[list[BaseMessage], add_messages]`, enabling automatic appending of tool responses and assistant utterances.
2. **`MemorySaver` Checkpointing**: Saves full conversational state per `thread_id`, empowering the agent to maintain context across multi-turn conversations (e.g., student asks about fees, and in the next question refers to "that fee" without repeating the program name).
3. **Strict Grounding System Prompt**: Instructs the agent to remain strictly within the HIST institutional boundaries, use the calculator for all arithmetic, and cite relevant sections.

In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage, SystemMessage
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver

# 1. State Definition
class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

# 2. Bind Tools to LLM
llm_with_tools = llm.bind_tools(tools)
tool_node = ToolNode(tools)

# 3. Production System Prompt
SYSTEM_PROMPT = """You are UniGuide AI, the official Senior Admissions Advisor at University of Education (UE).

Your Objectives:
- Provide accurate, clear, and professional guidance to prospective students and parents.
- Answer questions regarding degree programs, eligibility, fee structures, scholarships, admission tests (HAT), and university policies.

Core Ground Rules:
1. RAG Grounding: Always use `university_rag_search` for queries regarding HIST policies, fees, dates, and requirements. Do NOT invent or assume facts.
2. Mathematical Accuracy: Whenever computing fees, discounts, aggregate marks, or remaining balances, you MUST use the `safe_calculator` tool. Never do mental math.
3. Multi-Turn Context: Maintain continuity with previous student messages. If the student refers to 'it', 'that program', or 'the fee', use the conversation history to resolve the reference.
4. Out-of-Scope Handling: If asked about medical degrees (MBBS) or programs HIST does not offer, clearly state that HIST specializes in Computing and Business disciplines and does not offer those programs.
5. Tone: Polite, supportive, authoritative, and concise. Format lists with clear markdown bullet points.
"""

# 4. Agent Node
def agent_node(state: AgentState):
    messages = state["messages"]
    # Prepend system prompt if not already present
    if not isinstance(messages[0], SystemMessage):
        messages = [SystemMessage(content=SYSTEM_PROMPT)] + messages
    
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

# 5. Build Graph Workflow
workflow = StateGraph(AgentState)

workflow.add_node("agent", agent_node)
workflow.add_node("tools", tool_node)

workflow.add_edge(START, "agent")
workflow.add_conditional_edges("agent", tools_condition)
workflow.add_edge("tools", "agent")

# 6. Attach Conversational Memory Checkpointer
memory = MemorySaver()
app = workflow.compile(checkpointer=memory)

print("UniGuide AI LangGraph Agent compiled successfully with MemorySaver checkpointing!")

In [ ]:
# Display Graph Topology
try:
    from IPython.display import Image, display
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception:
    print("Graph ASCII Representation:")
    print(app.get_graph().draw_ascii())

## 🧪 Module 6: End-to-End Verification & Benchmark Suite
To ensure the agent is production-ready, we execute **4 comprehensive verification scenarios**:
1. **Scenario 1 (Single-Hop RAG)**: Information retrieval on BSCS & BSAI eligibility and admission test rules.
2. **Scenario 2 (Multi-Step Reasoning + Calculator)**: Tuition fee lookup for BSCS + calculation of a 50% Merit Scholarship.
3. **Scenario 3 (Conversational Memory Verification)**: Testing that the agent recalls earlier turns in a multi-turn dialogue using `thread_id`.
4. **Scenario 4 (Hallucination Guardrail Check)**: Asking about an unsupported degree (e.g. MBBS) to verify safe refusal.

In [ ]:
import time
from langchain_core.messages import HumanMessage

def query_agent(prompt: str, thread_id: str = "benchmark_session", verbose: bool = True):
    """Helper to invoke the agent with conversational memory tracking and auto-retry on TPM limits."""
    config = {"configurable": {"thread_id": thread_id}}
    
    if verbose:
        print(f"\n{'='*70}")
        print(f"[USER] (Session: {thread_id}):\n{prompt}")
        print(f"{'='*70}")
    
    # Auto-retry loop to handle Groq Free Tier TPM limits gracefully
    max_attempts = 4
    for attempt in range(max_attempts):
        try:
            result = app.invoke(
                {"messages": [HumanMessage(content=prompt)]},
                config=config
            )
            break
        except Exception as e:
            err_msg = str(e)
            if "429" in err_msg or "Rate limit" in err_msg or "tokens per minute" in err_msg:
                wait_seconds = 3 * (attempt + 1)
                print(f"[RATE LIMIT] Free-tier TPM limit approached. Waiting {wait_seconds}s before retrying (Attempt {attempt+1}/{max_attempts})...")
                time.sleep(wait_seconds)
            else:
                raise e
    else:
        raise TimeoutError("Exceeded max retry attempts for rate limits.")
    
    last_message = result["messages"][-1]
    
    if verbose:
        # Check if tools were executed
        tool_calls_found = False
        for msg in result["messages"]:
            if hasattr(msg, "tool_calls") and msg.tool_calls:
                for tc in msg.tool_calls:
                    print(f"[TOOL CALL]: {tc['name']} with args: {tc['args']}")
                    tool_calls_found = True
            elif msg.__class__.__name__ == "ToolMessage":
                print(f"[TOOL RESULT]: {msg.content[:150]}...")
                
        print(f"\n[UNIGUIDE AI]:\n{last_message.content}")
        print(f"{'-'*70}")
        
    return last_message.content

In [ ]:
import time
time.sleep(1) # Pace requests to respect free-tier TPM

# Benchmark 1: RAG Retrieval on Program Eligibility & Tests
response_1 = query_agent(
    prompt="What are the eligibility criteria for BS Computer Science and BSAI? Is there an entry test waiver?",
    thread_id="test_suite_session"
)

In [ ]:
import time
time.sleep(2) # Pace requests to respect free-tier TPM

# Benchmark 2: Multi-Step Reasoning (RAG Fee Retrieval + Safe Calculator)
response_2 = query_agent(
    prompt="What is the semester fee for BSCS at University of Education? If I qualify for the 50% Merit Scholarship, what will be my tuition fee?",
    thread_id="test_suite_session"
)

### 🔄 Scenario 3: Conversational Memory Continuity Verification
Here we send a **follow-up query** in the *same* session (`test_suite_session`).
Notice that the prompt refers to **"that program"** and **"sibling discount"** without specifying BSCS or the fee amount. The agent must recall the context from previous turns through `MemorySaver`!

In [ ]:
import time
time.sleep(2) # Pace requests to respect free-tier TPM

# Benchmark 3: Memory Persistence (Contextual Follow-up)
response_3 = query_agent(
    prompt="My younger sister is also enrolling at University of Education. What sibling discount applies, and what would my remaining fee be for that program with the sibling discount applied instead?",
    thread_id="test_suite_session"
)

In [ ]:
import time
time.sleep(2) # Pace requests to respect free-tier TPM

# Benchmark 4: Guardrail & Hallucination Resistance
response_4 = query_agent(
    prompt="Can I apply for an MBBS or BDS medical degree at University of Education for Fall 2026?",
    thread_id="test_suite_session"
)

## 💬 Module 7: Interactive Live Advising Console
Run the cell below to start an interactive chat session with UniGuide AI. Each session maintains continuous multi-turn memory under a unique `thread_id`. Type `exit` or `quit` to conclude the conversation.

In [ ]:
import uuid

def launch_interactive_advisor():
    session_id = f"student_{str(uuid.uuid4())[:8]}"
    print(f"UniGuide AI Admissions Advising Session Started (Session ID: {session_id})")
    print("Type your questions below. Enter 'exit' or 'quit' to terminate the session.\n")
    
    while True:
        try:
            user_input = input("You: ").strip()
            if not user_input:
                continue
            if user_input.lower() in ["exit", "quit", "q"]:
                print("\nThank you for consulting UniGuide AI. Best wishes for your academic journey at HIST!")
                break
            
            print("\nUniGuide AI is thinking...")
            query_agent(user_input, thread_id=session_id, verbose=True)
            print()
        except (KeyboardInterrupt, EOFError):
            print("\nSession closed.")
            break

# To start an interactive conversation, uncomment the line below:
# launch_interactive_advisor()